In [1]:

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cv2
import seaborn as sns
from matplotlib.image import imread
from PIL import Image
import tensorflow as tf
np.random.seed(1337)
import gc
from tensorflow.keras.utils import to_categorical
import tensorflow as tf

from tensorflow.keras import layers
from keras.callbacks import ReduceLROnPlateau
from keras.optimizers import Adam
from tensorflow.keras.models import Sequential,Model
from tensorflow.keras.layers import Input,Reshape,Multiply, Conv2DTranspose,Dropout, AveragePooling2D,Flatten, Dense, Conv2D,MaxPool2D, MaxPooling2D, BatchNormalization,concatenate,UpSampling2D
from tensorflow.keras.callbacks import ModelCheckpoint
import warnings
warnings.filterwarnings('ignore')


img_size = 128 # size of the image
dataset = os.listdir("music_dataset_spectro_full/train")
labels = dataset # use the folder paths to create the labels for the instruments
print(labels)

['Accordion', 'Acoustic_Guitar', 'Banjo', 'Bass_Guitar', 'Clarinet', 'cowbell', 'Dobro', 'Drum_set', 'Electric_Guitar', 'flute', 'Harmonium', 'Horn', 'Keyboard', 'Mandolin', 'Organ', 'Piano', 'Saxophone', 'Shakers', 'Tambourine', 'Trombone', 'Trumpet', 'Ukulele', 'vibraphone', 'Violin']


In [2]:
# code to make the mixes array
def get_mixes_array(data_dir):
    data = []
    path = os.path.join(data_dir+"mix") # create path for the folder
    for img in os.listdir(data_dir+"mix"): # loop through all images in mix
        for stem in os.listdir(data_dir+"stems"):# loop through the stems
            if img.split("_")[0] == stem.split("_")[0]: #check if the mix name matches the stem number
                temp=stem.split("_",1)
                label = temp[1].split(".")[0] # use splits to get the instrument label
                class_num = labels.index(label) 
                try:
                    img_arr = cv2.imread(os.path.join(path,img),0)# load image as greyscale
                    resized_arr = img_arr[:img_size,:img_size] # resize the image
                    data.append([resized_arr,class_num])
                    gc.collect()
                except Exception as e:
                    print(e)
    return np.array(data,dtype="object") 


In [3]:
#code to make the stems array
def get_stems_array(data_dir):
    data = []
    path = os.path.join(data_dir)
    for img in os.listdir(data_dir):
            try:
                img_arr = cv2.imread(os.path.join(path,img),0)
                resized_arr = img_arr[:img_size,:img_size]# resize image
                data.append([resized_arr])
                gc.collect()
            except Exception as e:
                print(e)
    return np.array(data,dtype="float32") 

In [4]:
x_train = get_mixes_array("Music_separation_dataset/train/")

y_train = get_stems_array("Music_separation_dataset/train/stems")

x_valid = get_mixes_array("Music_separation_dataset/valid/")

y_valid = get_stems_array("Music_separation_dataset/valid/stems")



KeyboardInterrupt: 

In [ ]:
x_train_mix=[]
x_train_label=[]

x_valid_mix=[]
x_valid_label=[]

for feature, label in x_train: #separate image and label 
    x_train_mix.append(feature)
    x_train_label.append(label)


for feature, label in x_valid:#separate image and label 
    x_valid_mix.append(feature)
    x_valid_label.append(label)

del x_train
del x_valid

In [ ]:
gc.collect()
x_train_mix = np.array(x_train_mix)/255 # normalize the spectrograms
gc.collect()
x_valid_mix = np.array(x_valid_mix)/255 # normalize the spectrograms
gc.collect()
y_train = np.array(y_train)/255 # normalize the spectrograms
gc.collect()
y_valid = np.array(y_valid)/255 # normalize the spectrograms
gc.collect()


0

In [ ]:
x_train_mix = x_train_mix.reshape(-1, img_size, img_size, 1)
y_train = y_train.reshape(-1, img_size, img_size, 1)

x_valid_mix = x_valid_mix.reshape(-1, img_size, img_size, 1)
y_valid = y_valid.reshape(-1, img_size, img_size, 1)


In [ ]:
x_train_label = np.array(x_train_label)
x_train_label = to_categorical(x_train_label)

x_valid_label = np.array(x_valid_label)
x_valid_label = to_categorical(x_valid_label)

In [ ]:
print(x_valid_label.shape) #see the shapes 
print(x_train_mix.shape)
print(x_train_label.shape)
print(y_train.shape)

(1265, 24)
(10492, 128, 128, 1)
(10492, 24)
(10492, 128, 128, 1)


In [ ]:
num_classes = 24
def Unet():
    inputs = Input(shape=(None,None,1))
    label_input = Input(shape=(num_classes,))

    conv1 = Conv2D(16, (3,3), activation = 'leaky_relu', padding='same')(inputs)
    conv1 = BatchNormalization()(conv1)
    conv1 = Conv2D(16, (3,3), activation = 'leaky_relu', padding='same')(conv1)
    conv1 = BatchNormalization()(conv1)
    pool1 = MaxPool2D((2,2))(conv1)

    conv2 = Conv2D(32, (3,3), activation = 'leaky_relu', padding='same')(pool1)
    conv2 = BatchNormalization()(conv2)
    conv2 = Conv2D(32, (3,3), activation = 'leaky_relu', padding='same')(conv2)
    conv2 = BatchNormalization()(conv2)
    pool2 = MaxPool2D((2,2))(conv2)

    conv3 = Conv2D(64, (3,3), activation = 'leaky_relu', padding='same')(pool2)
    conv3 = BatchNormalization()(conv3)
    conv3 = Conv2D(64, (3,3), activation = 'leaky_relu', padding='same')(conv3)
    conv3 = BatchNormalization()(conv3)
    pool3 = MaxPool2D((2,2))(conv3)

    conv4 = Conv2D(128, (3,3), activation = 'leaky_relu', padding='same')(pool3)
    conv4 = BatchNormalization()(conv4)
    conv4 = Conv2D(128, (3,3), activation = 'leaky_relu', padding='same')(conv4)
    conv4 = BatchNormalization()(conv4)
    pool4 = MaxPool2D((2,2))(conv4)

    conv5 = Conv2D(256, (3,3), activation = 'leaky_relu', padding='same')(pool4)
    conv5 = BatchNormalization()(conv5)
    conv5 = Conv2D(256, (3,3), activation = 'leaky_relu', padding='same')(conv5)
    conv5 = BatchNormalization()(conv5)
    
    label_lay = Dense(256,activation="relu")(label_input)
    label_lay = Reshape((1,1,256))(label_lay)
    label_lay = UpSampling2D((8,8))(label_lay)
    multi_bottle = concatenate([conv5,label_lay])
    
    goUp1 = Conv2DTranspose(128,(3,3),activation = 'leaky_relu',padding="same",strides=(2,2))(multi_bottle)
    goUp1 = BatchNormalization()(goUp1)
    goUp1 = concatenate([goUp1,conv4])
    conv6 = Conv2D(128, (3,3), activation = 'leaky_relu', padding='same')(goUp1)
    conv6 = BatchNormalization()(conv6)
    conv6 = Conv2D(128, (3,3), activation = 'leaky_relu', padding='same')(conv6)
    conv6 = BatchNormalization()(conv6)

    goUp2 = Conv2DTranspose(64,(3,3),activation = 'leaky_relu',padding="same",strides=(2,2))(conv6)
    goUp2 = BatchNormalization()(goUp2)
    goUp2 = concatenate([goUp2,conv3])
    conv7 = Conv2D(64, (3,3), activation = 'leaky_relu', padding='same')(goUp2)
    conv7 = BatchNormalization()(conv7)
    conv7 = Conv2D(64, (3,3), activation = 'leaky_relu', padding='same')(conv7)
    conv7 = BatchNormalization()(conv7)

    goUp3 = Conv2DTranspose(32,(3,3),activation = 'leaky_relu',padding="same",strides=(2,2))(conv7)
    goUp3 = BatchNormalization()(goUp3)
    goUp3 = concatenate([goUp3,conv2])
    conv8 = Conv2D(32, (3,3), activation = 'leaky_relu', padding='same')(goUp3)
    conv8 = BatchNormalization()(conv8)
    conv8 = Conv2D(32, (3,3), activation = 'leaky_relu', padding='same',)(conv8)
    conv8 = BatchNormalization()(conv8)

    goUp4 = Conv2DTranspose(16,(3,3),activation = 'leaky_relu',padding="same",strides=(2,2))(conv8)
    goUp4 = BatchNormalization()(goUp4)
    goUp4 = concatenate([goUp4,conv1])
    conv9 = Conv2D(16, (3,3), activation = 'leaky_relu', padding='same')(goUp4)
    conv9 = BatchNormalization()(conv9)
    conv9 = Conv2D(16, (3,3), activation = 'leaky_relu', padding='same')(conv9)
    conv9 = BatchNormalization()(conv9)


    outputs = Conv2D(1,(3,3),activation="sigmoid", padding="same")(conv9)

    model = Model(inputs=[inputs,label_input],outputs=[outputs])
    return model

adam_Opt = Adam(learning_rate=0.000001)

model = Unet()
model.compile(
              optimizer = adam_Opt, loss = "mse",
              metrics = ['mae']
              )
     

In [ ]:
model.summary()

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_2       │ (None, None,      │          0 │ -                 │
│ (InputLayer)        │ None, 1)          │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_19 (Conv2D)  │ (None, None,      │        160 │ input_layer_2[0]… │
│                     │ None, 16)         │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, None,      │         64 │ conv2d_19[0][0]   │
│ (BatchNormalizatio… │ None, 16)         │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_20 (Conv2D)  │ (None, None,      │      2,320 │ batch_normalizat… │
│                     │ None, 16)         │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, None,      │         64 │ conv2d_20[0][0]   │
│ (BatchNormalizatio… │ None, 16)         │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_4     │ (None, None,      │          0 │ batch_normalizat… │
│ (MaxPooling2D)      │ None, 16)         │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_21 (Conv2D)  │ (None, None,      │      4,640 │ max_pooling2d_4[… │
│                     │ None, 32)         │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, None,      │        128 │ conv2d_21[0][0]   │
│ (BatchNormalizatio… │ None, 32)         │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_22 (Conv2D)  │ (None, None,      │      9,248 │ batch_normalizat… │
│                     │ None, 32)         │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, None,      │        128 │ conv2d_22[0][0]   │
│ (BatchNormalizatio… │ None, 32)         │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_5     │ (None, None,      │          0 │ batch_normalizat… │
│ (MaxPooling2D)      │ None, 32)         │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_23 (Conv2D)  │ (None, None,      │     18,496 │ max_pooling2d_5[… │
│                     │ None, 64)         │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, None,      │        256 │ conv2d_23[0][0]   │
│ (BatchNormalizatio… │ None, 64)         │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_24 (Conv2D)  │ (None, None,      │     36,928 │ batch_normalizat… │
│                     │ None, 64)         │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, None,      │        256 │ conv2d_24[0][0]   │
│ (BatchNormalizatio… │ None, 64)         │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_6     │ (None, None,      │          0 │ batch_normalizat… │
│ (MaxPooling2D)      │ None, 64)         │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_25 (Conv2D)  │ (None, None,      │     73,856 │ max_pooling2d_6[

 Total params: 2,466,705 (9.41 MB)

 Trainable params: 2,463,281 (9.40 MB)

 Non-trainable params: 3,424 (13.38 KB)

In [ ]:
checkpoint = ModelCheckpoint('Checkpoint.keras',"val_loss", save_freq=4500,mode="min",verbose=1)


: 

In [ ]:
batch_size = 8
n_epochs = 300
model.fit(x=[x_train_mix,x_train_label], y=y_train, batch_size = batch_size,
                    epochs = n_epochs, validation_data = ([x_valid_mix,x_valid_label], y_valid),callbacks=[checkpoint],shuffle=True)

Epoch 1/400
1312/1312 ━━━━━━━━━━━━━━━━━━━━ 287s 210ms/step - loss: 0.0380 - mae: 0.1529 - val_loss: 0.0257 - val_mae: 0.1240
Epoch 2/400
1312/1312 ━━━━━━━━━━━━━━━━━━━━ 275s 209ms/step - loss: 0.0273 - mae: 0.1298 - val_loss: 0.0257 - val_mae: 0.1260
Epoch 3/400
1312/1312 ━━━━━━━━━━━━━━━━━━━━ 275s 209ms/step - loss: 0.0259 - mae: 0.1257 - val_loss: 0.0235 - val_mae: 0.1173
Epoch 4/400
 563/1312 ━━━━━━━━━━━━━━━━━━━━ 2:32 203ms/step - loss: 0.0258 - mae: 0.1248
Epoch 4: saving model to Checkpoint.keras

Epoch 4: finished saving model to Checkpoint.keras
1312/1312 ━━━━━━━━━━━━━━━━━━━━ 275s 210ms/step - loss: 0.0251 - mae: 0.1231 - val_loss: 0.0243 - val_mae: 0.1219
Epoch 5/400
1312/1312 ━━━━━━━━━━━━━━━━━━━━ 276s 210ms/step - loss: 0.0246 - mae: 0.1216 - val_loss: 0.0227 - val_mae: 0.1160
Epoch 6/400
1312/1312 ━━━━━━━━━━━━━━━━━━━━ 275s 210ms/step - loss: 0.0244 - mae: 0.1209 - val_loss: 0.0231 - val_mae: 0.1162
Epoch 7/400
1127/1312 ━━━━━━━━━━━━━━━━━━━━ 37s 204ms/step - loss: 0.0243 - mae: 

In [ ]:
gc.collect()

model.save('my_separator_model_2.keras')

In [ ]:
img = cv2.imread("Music_separation_dataset/test/mix/4044_mix.png",0)

img=img[:128,:128]
img= np.reshape(img,(-1, 128, 128, 1))
img=img/255
drum = np.array([[0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0,0,0,0,0,0,0,0,0,0,0,0]])

predimg= np.squeeze(model.predict([img,drum]))

prediction= predimg*255
print(prediction.shape)

cv2.imwrite("separated_drum_part2.png",prediction)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 383ms/step
(128, 128)


True

In [ ]:
import librosa

image = cv2.imread("separated_drum_part2.png",0)

to_db = (image.astype(np.float32)/255)*80-80

power = librosa.db_to_power(to_db)

audio = librosa.feature.inverse.mel_to_audio(power,n_fft = 2048, hop_length = 512,n_iter=256,sr=22050)


audio = librosa.util.normalize(audio)

import soundfile
soundfile.write('drum_prediction.wav',audio, 22050)




In [ ]:
img = cv2.imread("Music_separation_dataset/test/mix/4052_mix.png",0)

img=img[:128,:128]
img= np.reshape(img,(-1, 128, 128, 1))
img=img/255
acoustic = np.array([[0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,0,0,0,0,0,0,0,0,0,0,0]])

predimg= np.squeeze(model.predict([img,acoustic]))

prediction= predimg*255
print(prediction.shape)

cv2.imwrite("separated_part_acoustic2.png",prediction)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step
(128, 128)


True

In [ ]:
import librosa

image = cv2.imread("separated_part_acoustic2.png",0)

to_db = (image.astype(np.float32)/255)*80-80

power = librosa.db_to_power(to_db)

audio = librosa.feature.inverse.mel_to_audio(power,n_fft = 2048, hop_length = 512,n_iter=256,sr=22050)


audio = librosa.util.normalize(audio)

import soundfile
soundfile.write('acoustic_prediction2.wav',audio, 22050)

In [ ]:
img = cv2.imread("Music_separation_dataset/test/mix/4056_mix.png",0)

img=img[:128,:128]
img= np.reshape(img,(-1, 128, 128, 1))
img=img/255
acoustic = np.array([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,0,0,0,0,0,0,0,1,0,0,0]])

predimg= np.squeeze(model.predict([img,acoustic]))

prediction= predimg*255
print(prediction.shape)

cv2.imwrite("separated_part_Trumpet2.png",prediction)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 41ms/step
(128, 128)


True

In [ ]:
import librosa

image = cv2.imread("separated_part_Trumpet2.png",0)

to_db = (image.astype(np.float32)/255)*80-80

power = librosa.db_to_power(to_db)

audio = librosa.feature.inverse.mel_to_audio(power,n_fft = 2048, hop_length = 512,n_iter=256,sr=22050)


audio = librosa.util.normalize(audio)

import soundfile
soundfile.write('trumpet_prediction2.wav',audio, 22050)

In [ ]:
freq, sr = librosa.load("audio_separator_dataset/mix/0.wav")

image = cv2.imread("separated_drum_part2.png",0)

to_db = (image.astype(np.float32)/255)*80-80

power = librosa.db_to_power(to_db)

short_time= librosa.stft(freq)

mag,phase = librosa.magphase(short_time)

inverse_mel = librosa.feature.inverse.mel_to_stft(power,sr=sr)

phase = phase[:img_size,:img_size]

short_inverse = inverse_mel * np.exp(1j*phase)

recon = librosa.istft(short_inverse)
recon = librosa.util.normalize(recon)

soundfile.write('drum_pediction_Phase2.wav',recon, 22050)


In [ ]:
freq, sr = librosa.load("audio_separator_dataset/mix/462.wav")

image = cv2.imread("separated_part_acoustic2.png",0)


to_db = (image.astype(np.float32)/255)*80-80

power = librosa.db_to_power(to_db)

short_time= librosa.stft(freq)

mag,phase = librosa.magphase(short_time)

inverse_mel = librosa.feature.inverse.mel_to_stft(power,sr=sr)

phase = phase[:img_size,:img_size]

short_inverse = inverse_mel * np.exp(1j*phase)

recon = librosa.istft(short_inverse)
recon = librosa.util.normalize(recon)

soundfile.write('acoustic_pediction_Phase2.wav',recon, 22050)

In [ ]:
freq, sr = librosa.load("audio_separator_dataset/mix/19.wav")

image = cv2.imread("separated_part_trumpet2.png",0)


to_db = (image.astype(np.float32)/255)*80-80

power = librosa.db_to_power(to_db)

short_time= librosa.stft(freq)

mag,phase = librosa.magphase(short_time)

inverse_mel = librosa.feature.inverse.mel_to_stft(power,sr=sr)
phase = phase[:img_size,:img_size]


short_inverse = inverse_mel * np.exp(1j*phase)

recon = librosa.istft(short_inverse)
recon = librosa.util.normalize(recon)

soundfile.write('trumpet_pediction_Phase2.wav',recon, 22050)

In [ ]:
img = cv2.imread("Music_separation_dataset/test/mix/4048_mix.png",0)

img=img[:128,:128]
img= np.reshape(img,(-1, 128, 128, 1))
img=img/255
acoustic = np.array([[1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,0,0,0,0,0,0,0,0,0,0,0]])

predimg= np.squeeze(model.predict([img,acoustic]))

prediction= predimg*255
print(prediction.shape)

cv2.imwrite("separated_part_accodrian.png",prediction)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step
(128, 128)


True

In [ ]:
image = cv2.imread("separated_part_accodrian.png",0)

to_db = (image.astype(np.float32)/255)*80-80

power = librosa.db_to_power(to_db)

audio = librosa.feature.inverse.mel_to_audio(power,n_fft = 2048, hop_length = 512,n_iter=256,sr=22050)


audio = librosa.util.normalize(audio)


soundfile.write('separated_part_accordian.wav',audio, 22050)

In [ ]:
img = cv2.imread("Music_separation_dataset/test/mix/4045_mix.png",0)

img=img[:128,:128]
img= np.reshape(img,(-1, 128, 128, 1))
img=img/255
acoustic = np.array([[0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,0,0,0,0,0,0,0,0,0,0,0]])

predimg= np.squeeze(model.predict([img,acoustic]))

prediction= predimg*255
print(prediction.shape)

cv2.imwrite("separated_part_banjo.png",prediction)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 41ms/step
(128, 128)


True

In [ ]:
image = cv2.imread("separated_part_banjo.png",0)

to_db = (image.astype(np.float32)/255)*80-80

power = librosa.db_to_power(to_db)

audio = librosa.feature.inverse.mel_to_audio(power,n_fft = 2048, hop_length = 512,n_iter=256,sr=22050)


audio = librosa.util.normalize(audio)


soundfile.write('separated_part_banjo.wav',audio, 22050)

In [ ]:
img = cv2.imread("Music_separation_dataset/test/mix/4038_mix.png",0)

img=img[:128,:128]
img= np.reshape(img,(-1, 128, 128, 1))
img=img/255
acoustic = np.array([[0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0,0,0,0,0,0,0,0,0,0,0,0]])

predimg= np.squeeze(model.predict([img,acoustic]))

prediction= predimg*255
print(prediction.shape)

cv2.imwrite("separated_part_bass.png",prediction)


image = cv2.imread("separated_part_bass.png",0)

to_db = (image.astype(np.float32)/255)*80-80

power = librosa.db_to_power(to_db)

audio = librosa.feature.inverse.mel_to_audio(power,n_fft = 2048, hop_length = 512,n_iter=256,sr=22050)


audio = librosa.util.normalize(audio)


soundfile.write('separated_part_bass.wav',audio, 22050)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 63ms/step
(128, 128)


In [ ]:
img = cv2.imread("Music_separation_dataset/test/mix/4041_mix.png",0)

img=img[:128,:128]
img= np.reshape(img,(-1, 128, 128, 1))
img=img/255
acoustic = np.array([[0, 0, 0, 0,1, 0, 0, 0, 0, 0, 0, 0, 0,0,0,0,0,0,0,0,0,0,0,0]])

predimg= np.squeeze(model.predict([img,acoustic]))

prediction= predimg*255
print(prediction.shape)

cv2.imwrite("separated_part_clarinet.png",prediction)


image = cv2.imread("separated_part_clarinet.png",0)

to_db = (image.astype(np.float32)/255)*80-80

power = librosa.db_to_power(to_db)

audio = librosa.feature.inverse.mel_to_audio(power,n_fft = 2048, hop_length = 512,n_iter=256,sr=22050)


audio = librosa.util.normalize(audio)


soundfile.write('separated_part_clarinet.wav',audio, 22050)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step
(128, 128)


In [ ]:
img = cv2.imread("Music_separation_dataset/test/mix/4063_mix.png",0)

img=img[:128,:128]
img= np.reshape(img,(-1, 128, 128, 1))
img=img/255
acoustic = np.array([[0, 0, 0, 0,0, 1, 0, 0, 0, 0, 0, 0, 0,0,0,0,0,0,0,0,0,0,0,0]])

predimg= np.squeeze(model.predict([img,acoustic]))

prediction= predimg*255
print(prediction.shape)

cv2.imwrite("separated_part_cowbell.png",prediction)


image = cv2.imread("separated_part_cowbell.png",0)

to_db = (image.astype(np.float32)/255)*80-80

power = librosa.db_to_power(to_db)

audio = librosa.feature.inverse.mel_to_audio(power,n_fft = 2048, hop_length = 512,n_iter=256,sr=22050)


audio = librosa.util.normalize(audio)


soundfile.write('separated_part_cowbell.wav',audio, 22050)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step
(128, 128)


In [ ]:
img = cv2.imread("Music_separation_dataset/test/mix/4063_mix.png",0)

img=img[:128,:128]
img= np.reshape(img,(-1, 128, 128, 1))
img=img/255
acoustic = np.array([[0, 0, 0, 0,0, 0, 1, 0, 0, 0, 0, 0, 0,0,0,0,0,0,0,0,0,0,0,0]])

predimg= np.squeeze(model.predict([img,acoustic]))

prediction= predimg*255
print(prediction.shape)

cv2.imwrite("separated_part_dobro.png",prediction)


image = cv2.imread("separated_part_dobro.png",0)

to_db = (image.astype(np.float32)/255)*80-80

power = librosa.db_to_power(to_db)

audio = librosa.feature.inverse.mel_to_audio(power,n_fft = 2048, hop_length = 512,n_iter=256,sr=22050)


audio = librosa.util.normalize(audio)


soundfile.write('separated_part_dobro.wav',audio, 22050)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 43ms/step
(128, 128)


In [ ]:
img = cv2.imread("Music_separation_dataset/test/mix/4047_mix.png",0)

img=img[:128,:128]
img= np.reshape(img,(-1, 128, 128, 1))
img=img/255
acoustic = np.array([[0, 0, 0, 0,0, 0, 0, 0, 1, 0, 0, 0, 0,0,0,0,0,0,0,0,0,0,0,0]])

predimg= np.squeeze(model.predict([img,acoustic]))

prediction= predimg*255
print(prediction.shape)

cv2.imwrite("separated_part_eletric.png",prediction)


image = cv2.imread("separated_part_eletric.png",0)

to_db = (image.astype(np.float32)/255)*80-80

power = librosa.db_to_power(to_db)

audio = librosa.feature.inverse.mel_to_audio(power,n_fft = 2048, hop_length = 512,n_iter=256,sr=22050)


audio = librosa.util.normalize(audio)


soundfile.write('separated_part_eletric.wav',audio, 22050)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 42ms/step
(128, 128)


In [ ]:
img = cv2.imread("Music_separation_dataset/test/mix/4046_mix.png",0)

img=img[:128,:128]
img= np.reshape(img,(-1, 128, 128, 1))
img=img/255
acoustic = np.array([[0, 0, 0, 0,0, 0, 0,0, 0, 1, 0, 0, 0,0,0,0,0,0,0,0,0,0,0,0]])

predimg= np.squeeze(model.predict([img,acoustic]))

prediction= predimg*255
print(prediction.shape)

cv2.imwrite("separated_part_flute.png",prediction)


image = cv2.imread("separated_part_flute.png",0)

to_db = (image.astype(np.float32)/255)*80-80

power = librosa.db_to_power(to_db)

audio = librosa.feature.inverse.mel_to_audio(power,n_fft = 2048, hop_length = 512,n_iter=256,sr=22050)


audio = librosa.util.normalize(audio)


soundfile.write('separated_part_flute.wav',audio, 22050)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step
(128, 128)


In [ ]:
img = cv2.imread("Music_separation_dataset/test/mix/4063_mix.png",0)

img=img[:128,:128]
img= np.reshape(img,(-1, 128, 128, 1))
img=img/255
acoustic = np.array([[0, 0, 0, 0,0, 0, 0,0, 0, 0, 0, 0, 0,0,0,0,0,0,1,0,0,0,0,0]])

predimg= np.squeeze(model.predict([img,acoustic]))

prediction= predimg*255
print(prediction.shape)

cv2.imwrite("separated_part_tambourine.png",prediction)


image = cv2.imread("separated_part_tambourine.png",0)

to_db = (image.astype(np.float32)/255)*80-80

power = librosa.db_to_power(to_db)

audio = librosa.feature.inverse.mel_to_audio(power,n_fft = 2048, hop_length = 512,n_iter=256,sr=22050)


audio = librosa.util.normalize(audio)


soundfile.write('separated_part_tambourine.wav',audio, 22050)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step
(128, 128)
